In [1]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip -q install torchcde pyyaml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.8 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2: Setup Environment
# ============================================================
import os, random, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import yaml

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
GPU: Tesla T4


In [3]:
# ============================================================
# CELL 3: Clone Our Modified Repository
# ============================================================
os.chdir("/kaggle/working")
if not os.path.exists("PFE_Dynamic_Adj"):
    !git clone https://github.com/chemsou7876/PFE_Dynamic_Adj.git
os.chdir("/kaggle/working/PFE_Dynamic_Adj/PriSTI_modified")
!pwd

Cloning into 'PFE_Dynamic_Adj'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 54 (delta 4), reused 54 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 10.61 MiB | 26.49 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/kaggle/working/PFE_Dynamic_Adj/PriSTI_modified


In [4]:
# ============================================================
# CELL 4: Verify Project Structure
# ============================================================
print("=== Project Structure ===")
for f in ['layers.py', 'diff_models.py', 'main_model.py', 'generate_adj.py',
          'dataset_aqi36.py', 'utils.py', 'config/base.yaml']:
    exists = os.path.exists(f)
    print(f"  {'✅' if exists else '❌'} {f}")

# Check extracted data
extracted_dir = "../extracted_data"
for f in ['A_static.npy', 'var_mean.npy', 'var_std.npy']:
    path = os.path.join(extracted_dir, f)
    exists = os.path.exists(path)
    if exists:
        data = np.load(path)
        print(f"  ✅ {f} — shape: {data.shape}")
    else:
        print(f"  ❌ {f} — MISSING")

# Check dataset
data_path = "./data/pm25/SampleData/pm25_ground.txt"
if os.path.exists(data_path):
    df = pd.read_csv(data_path, index_col="datetime", parse_dates=True)
    print(f"  ✅ Dataset: {df.shape}")
else:
    print(f"  ❌ Dataset not found")

=== Project Structure ===
  ✅ layers.py
  ✅ diff_models.py
  ✅ main_model.py
  ✅ generate_adj.py
  ✅ dataset_aqi36.py
  ✅ utils.py
  ✅ config/base.yaml
  ✅ A_static.npy — shape: (36, 36)
  ✅ var_mean.npy — shape: (36,)
  ✅ var_std.npy — shape: (36,)
  ✅ Dataset: (8759, 36)


In [5]:
# ============================================================
# CELL 5: Global Seed
# ============================================================
GLOBAL_SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(GLOBAL_SEED)
print(f"✓ Seed: {GLOBAL_SEED}")


✓ Seed: 42


In [6]:
# ============================================================
# CELL 6: Configuration — SAME as PriSTI Original
# ============================================================
EVAL_LENGTH = 36
TARGET_DIM = 36
VAL_LEN = 0.1

BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-3
VALID_INTERVAL = 25

D_MODEL = 64
N_LAYERS = 4
N_HEADS = 8
DIFFUSION_STEPS = 100

NSAMPLE = 50
NUM_RUNS = 8
SEEDS = [42, 123, 456, 789, 1024, 2048, 3141, 5926]

MAX_VAL_BATCHES = None
MAX_TEST_BATCHES = None

# Dynamic adjacency parameters (NEW)
ALPHA = 0.6
BETA = 0.4

print(f"🎯 DYNAMIC ADJACENCY EXPERIMENT")
print(f"=" * 70)
print(f"SAME CONFIG AS PRISTI BASELINE:")
print(f"  Runs: {NUM_RUNS}, Epochs: {EPOCHS}, Batch: {BATCH_SIZE}")
print(f"  Model: {N_LAYERS} layers, {D_MODEL} channels, {N_HEADS} heads")
print(f"  Diffusion: {DIFFUSION_STEPS} steps, {NSAMPLE} samples")
print(f"")
print(f"NEW — DYNAMIC ADJACENCY:")
print(f"  α (spatial prior): {ALPHA}")
print(f"  β (local correlation): {BETA}")
print(f"  Data dir: ../extracted_data/")
print(f"=" * 70)


🎯 DYNAMIC ADJACENCY EXPERIMENT
SAME CONFIG AS PRISTI BASELINE:
  Runs: 8, Epochs: 25, Batch: 32
  Model: 4 layers, 64 channels, 8 heads
  Diffusion: 100 steps, 50 samples

NEW — DYNAMIC ADJACENCY:
  α (spatial prior): 0.6
  β (local correlation): 0.4
  Data dir: ../extracted_data/


In [7]:
# ============================================================
# CELL 7: Import Our Modified Modules
# ============================================================
import sys
sys.path.insert(0, '/kaggle/working/PFE_Dynamic_Adj/PriSTI_modified')
sys.path.insert(0, '/kaggle/working/PFE_Dynamic_Adj/layer2_dynamic')

from dataset_aqi36 import get_dataloader
from main_model import PriSTI_aqi36

print("✓ Modified PriSTI modules imported successfully")

# Quick test: verify DynamicAdjacency loads
from dynamic_adjacency import DynamicAdjacency
test_dyn = DynamicAdjacency(
    data_dir="/kaggle/working/PFE_Dynamic_Adj/extracted_data",
    alpha=ALPHA, beta=BETA, device="cpu"
)
print(f"✓ DynamicAdjacency OK — A_static non-zero: {(test_dyn.A_static > 0).sum().item()}")
del test_dyn


✓ Modified PriSTI modules imported successfully
✅ DynamicAdjacency initialisée
   36 capteurs, α=0.6, β=0.4
   A_static non-zero: 686
✓ DynamicAdjacency OK — A_static non-zero: 686


In [8]:
# ============================================================
# CELL 8: Create Configuration
# ============================================================
config = {
    "model": {
        "is_unconditional": False,
        "timeemb": 128,
        "featureemb": 16,
        "target_strategy": "hybrid",
        "use_guide": True,
        "mask_sensor": []
    },
    "diffusion": {
        "layers": N_LAYERS,
        "channels": D_MODEL,
        "nheads": N_HEADS,
        "diffusion_embedding_dim": 128,
        "beta_start": 0.0001,
        "beta_end": 0.2,
        "num_steps": DIFFUSION_STEPS,
        "schedule": "quad",
        "is_adp": True,
        "proj_t": 16,
        "is_cross_t": True,
        "is_cross_s": True,
        "adj_file": "AQI36",
        "device": device,
        # NEW: Dynamic adjacency parameters
        "alpha": ALPHA,
        "beta": BETA,
        "dynamic_adj_data_dir": "/kaggle/working/PFE_Dynamic_Adj/extracted_data",
    },
    "train": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "valid_epoch_interval": VALID_INTERVAL,
        "is_lr_decay": True
    },
    "seed": GLOBAL_SEED
}

print("✓ Configuration created")


✓ Configuration created


In [9]:
# ============================================================
# CELL 9: Training Function (IDENTICAL to PriSTI notebook)
# ============================================================
def train_one_run(model, train_loader, valid_loader, config, run_id, seed,
                  scaler, mean_scaler, max_val_batches=10):
    import time

    optimizer = Adam(model.parameters(), lr=config["train"]["lr"], weight_decay=1e-6)

    p1 = int(0.75 * config["train"]["epochs"])
    p2 = int(0.9 * config["train"]["epochs"])
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[p1, p2], gamma=0.1)

    best_val_loss = float('inf')
    model_path = f"/kaggle/working/dynamic_adj_run{run_id}_seed{seed}.pth"

    for epoch in range(1, config["train"]["epochs"] + 1):
        model.train()
        train_loss, n_batches = 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{config['train']['epochs']}", leave=False)
        for batch in pbar:
            optimizer.zero_grad()
            loss = model(batch, is_train=1)

            if torch.isnan(loss) or torch.isinf(loss):
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        scheduler.step()
        avg_train = train_loss / max(n_batches, 1)

        if epoch % config["train"]["valid_epoch_interval"] == 0:
            model.eval()
            val_loss, n_val = 0, 0

            with torch.no_grad():
                for i, batch in enumerate(valid_loader):
                    if max_val_batches is not None and i >= max_val_batches:
                        break
                    loss = model(batch, is_train=0)
                    if not (torch.isnan(loss) or torch.isinf(loss)):
                        val_loss += loss.item()
                        n_val += 1

            avg_val = val_loss / max(n_val, 1)

            if avg_val < best_val_loss:
                best_val_loss = avg_val
                torch.save(model.state_dict(), model_path)
                marker = " ★"
            else:
                marker = ""

            print(f"\n  {'='*70}")
            print(f"  Epoch {epoch:02d}/{config['train']['epochs']}{marker}")
            print(f"  {'='*70}")
            print(f"  Train Loss: {avg_train:.10f}")
            print(f"  Val Loss:   {avg_val:.10f}")
            print(f"  {'='*70}")
        else:
            print(f"  Epoch {epoch:02d}/{config['train']['epochs']} | Train Loss: {avg_train:.10f}")

    return best_val_loss, model_path

print("✓ Training function ready")


✓ Training function ready


In [10]:
# ============================================================
# CELL 10: Evaluation Function (IDENTICAL to PriSTI notebook)
# ============================================================
def calc_quantile_CRPS(target, forecast, eval_points, mean_scaler, scaler):
    target = target * scaler + mean_scaler
    forecast = forecast * scaler + mean_scaler

    quantiles = np.arange(0.05, 1.0, 0.05)
    denom = torch.sum(torch.abs(target * eval_points))
    CRPS = 0

    for i in range(len(quantiles)):
        q_pred = []
        for j in range(len(forecast)):
            q_pred.append(torch.quantile(forecast[j : j + 1], quantiles[i], dim=1))
        q_pred = torch.cat(q_pred, 0)
        q_loss = 2 * torch.sum(
            torch.abs((q_pred - target) * eval_points * ((target <= q_pred) * 1.0 - quantiles[i]))
        )
        CRPS += q_loss / denom

    return CRPS.item() / len(quantiles)

def evaluate_model(model, test_loader, nsample, scaler, mean_scaler, max_batches=None):
    model.eval()
    mse_total, mae_total, evalpoints_total = 0, 0, 0

    all_target, all_evalpoint, all_samples = [], [], []

    with torch.no_grad():
        for i, batch in enumerate(tqdm(test_loader, desc="Evaluating", leave=False)):
            if max_batches is not None and i >= max_batches:
                break

            samples, c_target, eval_points, _, _ = model.evaluate(batch, nsample)

            samples = samples.permute(0, 1, 3, 2)
            c_target = c_target.permute(0, 2, 1)
            eval_points = eval_points.permute(0, 2, 1)

            samples_median = samples.median(dim=1).values

            mse_current = (((samples_median - c_target) * eval_points) ** 2) * (scaler ** 2)
            mae_current = (torch.abs((samples_median - c_target) * eval_points)) * scaler

            mse_total += mse_current.sum().item()
            mae_total += mae_current.sum().item()
            evalpoints_total += eval_points.sum().item()

            all_target.append(c_target)
            all_evalpoint.append(eval_points)
            all_samples.append(samples)

    rmse = np.sqrt(mse_total / evalpoints_total)
    mae = mae_total / evalpoints_total

    all_target = torch.cat(all_target, dim=0)
    all_evalpoint = torch.cat(all_evalpoint, dim=0)
    all_samples = torch.cat(all_samples, dim=0)

    mask = all_evalpoint > 0.5
    if mask.sum() > 0:
        pred_median = all_samples.median(dim=1).values
        y_true = all_target[mask].cpu().numpy()
        y_pred = pred_median[mask].cpu().numpy()
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
    else:
        r2 = 0.0

    crps = calc_quantile_CRPS(all_target, all_samples, all_evalpoint, mean_scaler, scaler)

    return rmse, mae, r2, crps

print("✓ Evaluation function ready")



✓ Evaluation function ready


In [11]:
# ============================================================
# CELL 11: MAIN EXPERIMENT
# ============================================================
import time

RESULTS_DIR = '/kaggle/working/dynamic_adj_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

RESULTS_FILE = os.path.join(RESULTS_DIR, 'experiment_summary.json')
results = {'test_rmse': [], 'test_mae': [], 'test_r2': [], 'test_crps': []}
all_runs_data = []

# PriSTI baseline for comparison
PRISTI_BASELINE = {
    'rmse': {'mean': 19.3217, 'std': 0.2787},
    'mae': {'mean': 10.1744, 'std': 0.0939},
    'r2': {'mean': 0.9130, 'std': 0.0028},
    'crps': {'mean': 0.1145, 'std': 0.0012}
}

print("\n" + "=" * 70)
print(f"🚀 DYNAMIC ADJACENCY EXPERIMENT: {NUM_RUNS} RUNS × {EPOCHS} EPOCHS")
print(f"   α={ALPHA}, β={BETA}")
print("=" * 70)

experiment_start = time.time()

for run_id, seed in enumerate(SEEDS, 1):
    print("\n" + "=" * 70)
    print(f"RUN {run_id}/{NUM_RUNS} - SEED {seed}")
    print("=" * 70)

    run_start = time.time()
    set_seed(seed)

    # Load data
    print("\n📦 Loading data...")
    train_loader, valid_loader, test_loader, scaler, mean_scaler = get_dataloader(
        batch_size=BATCH_SIZE,
        device=device,
        val_len=VAL_LEN,
        is_interpolate=config["model"]["use_guide"],
        num_workers=2,
        target_strategy=config["model"]["target_strategy"],
        mask_sensor=config["model"]["mask_sensor"]
    )
    print(f"✓ Train: {len(train_loader)}, Valid: {len(valid_loader)}, Test: {len(test_loader)}")

    # Create model
    print("\n🏗️ Building model...")
    model = PriSTI_aqi36(config, device, target_dim=TARGET_DIM, seq_len=EVAL_LENGTH).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✓ Parameters: {n_params:,}")

    # Train
    print(f"\n🚀 Training...")
    best_val, model_path = train_one_run(
        model, train_loader, valid_loader, config, run_id, seed,
        scaler, mean_scaler, max_val_batches=MAX_VAL_BATCHES
    )
    print(f"✓ Best val loss: {best_val:.10f}")

    # Load best model
    model.load_state_dict(torch.load(model_path))

    # Evaluate
    print(f"\n📊 Testing...")
    rmse, mae, r2, crps = evaluate_model(
        model, test_loader, NSAMPLE, scaler, mean_scaler, max_batches=MAX_TEST_BATCHES
    )

    run_time = time.time() - run_start

    # Display results with comparison
    print(f"\n{'=' * 70}")
    print(f"RUN {run_id} RESULTS (vs PriSTI baseline):")
    print(f"{'=' * 70}")
    rmse_diff = rmse - PRISTI_BASELINE['rmse']['mean']
    mae_diff = mae - PRISTI_BASELINE['mae']['mean']
    r2_diff = r2 - PRISTI_BASELINE['r2']['mean']
    crps_diff = crps - PRISTI_BASELINE['crps']['mean']
    print(f"  RMSE: {rmse:.4f}  (baseline: {PRISTI_BASELINE['rmse']['mean']:.4f}, diff: {rmse_diff:+.4f})")
    print(f"  MAE:  {mae:.4f}  (baseline: {PRISTI_BASELINE['mae']['mean']:.4f}, diff: {mae_diff:+.4f})")
    print(f"  R²:   {r2:.4f}  (baseline: {PRISTI_BASELINE['r2']['mean']:.4f}, diff: {r2_diff:+.4f})")
    print(f"  CRPS: {crps:.4f}  (baseline: {PRISTI_BASELINE['crps']['mean']:.4f}, diff: {crps_diff:+.4f})")
    print(f"  Time: {run_time/60:.1f} min")
    print(f"{'=' * 70}\n")

    results['test_rmse'].append(float(rmse))
    results['test_mae'].append(float(mae))
    results['test_r2'].append(float(r2))
    results['test_crps'].append(float(crps))

    run_data = {
        'run_id': run_id,
        'seed': seed,
        'metrics': {'rmse': float(rmse), 'mae': float(mae), 'r2': float(r2), 'crps': float(crps)},
        'training': {'best_val_loss': float(best_val), 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'learning_rate': LR},
        'model': {'parameters': n_params, 'layers': N_LAYERS, 'channels': D_MODEL, 'heads': N_HEADS, 'diffusion_steps': DIFFUSION_STEPS},
        'dynamic_adj': {'alpha': ALPHA, 'beta': BETA},
        'runtime': {'minutes': round(run_time / 60, 2)}
    }
    all_runs_data.append(run_data)

    run_file = os.path.join(RESULTS_DIR, f'run_{run_id}_seed_{seed}.json')
    with open(run_file, 'w') as f:
        json.dump(run_data, f, indent=2)

    # Save summary after each run
    experiment_summary = {
        'experiment_info': {
            'model': 'PriSTI + Dynamic Quality-Aware Adjacency',
            'total_runs': NUM_RUNS,
            'completed_runs': run_id,
            'seeds': SEEDS,
            'dynamic_adj': {'alpha': ALPHA, 'beta': BETA},
            'status': 'in_progress' if run_id < NUM_RUNS else 'completed'
        },
        'pristi_baseline': PRISTI_BASELINE,
        'our_results': {
            'rmse': {'values': results['test_rmse'], 'mean': float(np.mean(results['test_rmse'])), 'std': float(np.std(results['test_rmse']))},
            'mae': {'values': results['test_mae'], 'mean': float(np.mean(results['test_mae'])), 'std': float(np.std(results['test_mae']))},
            'r2': {'values': results['test_r2'], 'mean': float(np.mean(results['test_r2'])), 'std': float(np.std(results['test_r2']))},
            'crps': {'values': results['test_crps'], 'mean': float(np.mean(results['test_crps'])), 'std': float(np.std(results['test_crps']))}
        },
        'individual_runs': all_runs_data
    }
    with open(RESULTS_FILE, 'w') as f:
        json.dump(experiment_summary, f, indent=2)

    # Clean up GPU memory
    del model
    torch.cuda.empty_cache()

experiment_time = time.time() - experiment_start



🚀 DYNAMIC ADJACENCY EXPERIMENT: 8 RUNS × 25 EPOCHS
   α=0.6, β=0.4

RUN 1/8 - SEED 42

📦 Loading data...
✓ Train: 165, Valid: 5, Test: 3

🏗️ Building model...
✅ DynamicAdjacency initialisée
   36 capteurs, α=0.6, β=0.4
   A_static non-zero: 686
✓ Parameters: 689,090

🚀 Training...


  Epoch 01/25 | Train Loss: 0.3197342174


  Epoch 02/25 | Train Loss: 0.2218657353


  Epoch 03/25 | Train Loss: 0.1791136127


  Epoch 04/25 | Train Loss: 0.1783598307


...

  Epoch 22/25 | Train Loss: 0.1412039275


  Epoch 23/25 | Train Loss: 0.1350167562


  Epoch 24/25 | Train Loss: 0.1374740407



  Epoch 25/25 ★
  Train Loss: 0.1372986970
  Val Loss:   0.1499281853
✓ Best val loss: 0.1499281853

📊 Testing...



RUN 8 RESULTS (vs PriSTI baseline):
  RMSE: 19.3702  (baseline: 19.3217, diff: +0.0485)
  MAE:  9.9518  (baseline: 10.1744, diff: -0.2226)
  R²:   0.9112  (baseline: 0.9130, diff: -0.0018)
  CRPS: 0.1121  (baseline: 0.1145, diff: -0.0024)
  Time: 60.5 min



In [12]:
# ============================================================
# CELL 12: FINAL COMPARISON
# ============================================================
print("\n" + "=" * 70)
print("📊 EXPERIMENT COMPLETE — FINAL COMPARISON")
print("=" * 70)

print(f"\n{'Metric':<8} {'PriSTI (baseline)':<25} {'Ours (dynamic adj)':<25} {'Δ':<15} {'Better?'}")
print("-" * 80)

metrics = ['rmse', 'mae', 'r2', 'crps']
metric_names = ['RMSE', 'MAE', 'R²', 'CRPS']

for name, key in zip(metric_names, metrics):
    base_mean = PRISTI_BASELINE[key]['mean']
    base_std = PRISTI_BASELINE[key]['std']
    our_mean = float(np.mean(results[f'test_{key}']))
    our_std = float(np.std(results[f'test_{key}']))
    diff = our_mean - base_mean

    if key == 'r2':
        better = "✅ YES" if diff > 0 else "❌ NO"
    else:
        better = "✅ YES" if diff < 0 else "❌ NO"

    print(f"{name:<8} {base_mean:.4f} ± {base_std:.4f}         {our_mean:.4f} ± {our_std:.4f}         {diff:+.4f}         {better}")

print("-" * 80)

print(f"\n📊 PER-RUN BREAKDOWN:")
print("=" * 70)
for run_data in all_runs_data:
    print(f"Run {run_data['run_id']} (seed={run_data['seed']}): "
          f"RMSE={run_data['metrics']['rmse']:.4f}, "
          f"MAE={run_data['metrics']['mae']:.4f}, "
          f"R²={run_data['metrics']['r2']:.4f}, "
          f"CRPS={run_data['metrics']['crps']:.4f}")
print("=" * 70)

print(f"\n⏱️  Total: {experiment_time/60:.1f} min ({experiment_time/3600:.1f} hrs)")
print(f"⏱️  Per run: {experiment_time/60/NUM_RUNS:.1f} min")
print(f"\n📁 Results saved to: {RESULTS_DIR}")
print("=" * 70)


📊 EXPERIMENT COMPLETE — FINAL COMPARISON

Metric   PriSTI (baseline)         Ours (dynamic adj)        Δ               Better?
--------------------------------------------------------------------------------
RMSE     19.3217 ± 0.2787         18.8817 ± 0.2725         -0.4400         ✅ YES
MAE      10.1744 ± 0.0939         9.8463 ± 0.0770         -0.3281         ✅ YES
R²       0.9130 ± 0.0028         0.9169 ± 0.0028         +0.0039         ✅ YES
CRPS     0.1145 ± 0.0012         0.1109 ± 0.0009         -0.0036         ✅ YES
--------------------------------------------------------------------------------

📊 PER-RUN BREAKDOWN:
Run 1 (seed=42): RMSE=18.7122, MAE=9.8351, R²=0.9189, CRPS=0.1108
Run 2 (seed=123): RMSE=18.6770, MAE=9.7664, R²=0.9186, CRPS=0.1101
Run 3 (seed=456): RMSE=19.0058, MAE=9.9545, R²=0.9170, CRPS=0.1122
Run 4 (seed=789): RMSE=18.7840, MAE=9.8234, R²=0.9182, CRPS=0.1110
Run 5 (seed=1024): RMSE=18.7279, MAE=9.7549, R²=0.9184, CRPS=0.1101
Run 6 (seed=2048): RMSE=18.5444, M